In [42]:
import gym 
import wandb
import bauwerk
import numpy as np
from cfgs.parser import parse_cfg
from agent.sac import Agent as SAC

wandb.init(project="bauwerk", entity="enjeeneer")

In [2]:
env = gym.make("bauwerk/SolarBatteryHouse-v0")

In [29]:
obs_dim = 0
for key, value in env.observation_space.items():
    obs_dim += value.shape[0]
obs_dim

5

In [86]:
obs[0].load

AttributeError: 'dict' object has no attribute 'load'

wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
ERROR:root:dropped chunk 404 Client Error: Not Found for url: https://api.wandb.ai/files/enjeeneer/bauwerk/k26yz6fw/file_stream
NoneType: None
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
ERROR:root:dropped chunk 404 Client Error: Not Found for url: https://api.wandb.ai/files/enjeeneer/bauwerk/k26yz6fw/file_stream
NoneType: None
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
ERROR:root:dropped chunk 404 Client Error: Not Found for url: https://api.wandb.ai/files/enjeeneer/bauwerk/k26yz6fw/file_stream
NoneType: None
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
ERROR:root:dropped chunk 404 Client Error: Not Found for url: https://api.wandb.ai/files/enjeeneer/bauwerk/k26yz6fw/file_stream
NoneType: None
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
ERROR:root:dropped chunk 404 Client Error: Not Found for url: h

In [83]:
obs = env.reset(seed=2)
test = []
for _, value in obs[0].items():
    test.append(value)
np.concatenate(test, axis=0, dtype=np.float32)

array([0.704, 0.   , 0.   , 1.   , 0.   ], dtype=float32)

wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
ERROR:root:dropped chunk 404 Client Error: Not Found for url: https://api.wandb.ai/files/enjeeneer/bauwerk/k26yz6fw/file_stream
NoneType: None
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
ERROR:root:dropped chunk 404 Client Error: Not Found for url: https://api.wandb.ai/files/enjeeneer/bauwerk/k26yz6fw/file_stream
NoneType: None
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
ERROR:root:dropped chunk 404 Client Error: Not Found for url: https://api.wandb.ai/files/enjeeneer/bauwerk/k26yz6fw/file_stream
NoneType: None
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
ERROR:root:dropped chunk 404 Client Error: Not Found for url: https://api.wandb.ai/files/enjeeneer/bauwerk/k26yz6fw/file_stream
NoneType: None
wandb: ERROR Dropped streaming file chunk (see wandb/debug-internal.log)
ERROR:root:dropped chunk 404 Client Error: Not Found for url: h

In [7]:
obs = env.reset()
for _ in range(10):
    action = env.action_space.sample()
    print(action)
    obs, reward, terminated, truncated, info = env.step(action)
    print(obs)
    print(reward)
    print(terminated)
    print(truncated)
    print(info)
    print('\n')

env.close()

[-0.438]
{'load': array([0.], dtype=float32), 'pv_gen': array([0.], dtype=float32), 'battery_cont': array([0.], dtype=float32), 'time_of_day': array([0.966, 0.259])}
-0.17602874338626862
False
False
{'net_load': array([0.704], dtype=float32), 'charging_power': 0.0, 'cost': array([0.176], dtype=float32), 'battery_cont': array([0.], dtype=float32), 'price_threshold': 4.0}


[0.317]
{'load': array([0.29], dtype=float32), 'pv_gen': array([0.], dtype=float32), 'battery_cont': array([2.366], dtype=float32), 'time_of_day': array([0.866, 0.5  ])}
-0.5949408411979675
False
False
{'net_load': array([2.38], dtype=float32), 'charging_power': 2.379763349890709, 'cost': array([0.595], dtype=float32), 'battery_cont': array([2.366], dtype=float32), 'price_threshold': 4.0}


[0.444]
{'load': array([0.014], dtype=float32), 'pv_gen': array([0.], dtype=float32), 'battery_cont': array([5.677], dtype=float32), 'time_of_day': array([0.707, 0.707])}
-0.9050900340080261
False
False
{'net_load': array([3.62], d

In [ ]:
cfg = parse_cfg()
agent = SAC(cfg=cfg, env=env, obs_dim=env.observation_space.shape[0], act_dim=env.action_space.shape[0], models_dir='tmp/')

episodes = 100
eval_interal = 10
eval_episodes = 10

for i in range(episodes):
    ep_reward = 0
    episode_steps = 0
    evals = 0
    done = False
    state = env.reset()
    while not done:
        action = agent.act(state, evaluate=False)
        state_, reward, done, _ = env.step(action)
        agent.memory.store_transition(state, state_, action, reward, done)
        state = state_
        agent.n_steps += 1
        episode_steps += 1
        ep_reward += reward

        if agent.n_steps > agent.batch_size:
            value_loss, actor_loss, critic_loss = agent.learn()
            # wandb.log({
            #     'value_loss': value_loss,
            #     'actor_loss': actor_loss,
            #     'critic_loss': critic_loss
            # })

    print('Episode: {}, episode timesteps: {}, episode reward: {}'.format(i, episode_steps, ep_reward))

    if i % eval_interal == 0:
        eval_rewards = 0
        for i in range(eval_episodes):
            evals += 1
            done = False
            state = env.reset()
            while not done:
                action = agent.act(state, evaluate=True)
                state_, reward, done, _ = env.step(action)
                state = state_
                eval_rewards += reward

        eval_rewards = eval_rewards / eval_episodes # mean


        print('EVAL Episode: {}, episode reward: {}'.format(evals, eval_rewards))

    wandb.log({
        'train_reward': ep_reward,
        'test_reward': eval_rewards,
    })